In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType, StringType

## Generate date Range from 2019-01-01 to 2025-12-31
start_date = "2024-01-01"
end_date = "2025-12-31"

## create a dataframe with the dates from start to end
date_range = spark.sql(f"""
    SELECT explode(sequence(to_date('{start_date}'), to_date('{end_date}'), INTERVAL 1 DAY)) as Date
""")

StatementMeta(, fe2a1b6c-2d77-43cc-84fa-731de768968c, 4, Finished, Available, Finished, False)

In [ ]:
# add required transformations
calendar_df = date_range.withColumn("Day_Number",dayofmonth(col(Date))) \
    .withColumn("Day_Name",date_format(col("Date"), "EEEE")) \
    .withColumn("HOUR", lit(None).cast(IntegerType())) \
    .withColumn("Workday_Seq",expr("row_number() over (order by Date)").cast(IntegerType()) + 999) \
    .withColumn("Week_Of_Month",ceil((dayofyear(col("Date")) - dayofyear(date_trunc("month",col("Date")))) / 7) + 1)  \
    .withColumn("Week_Of_Year",weekofyear(col("Date"))) \
    .withColumn("Week_Start_Date",date_format(expr("date_sub(Date, dayofweek(Date) - 1)"), "yyyy-MM-dd")) \
    .withColumn("Month_Name",date_format(col("Date"), "MMMM")) \
    .withColumn("Month_Number",month(col("Date"))) \
    .withColumn("Month_Seq",expr("row_number() over (partition by month(Date) order by Date) - 1")) \
    .withColumn("Month_Year",date_format(col("Date"), "yyyy-MM")) \
    .withColumn("Month_Year_Number",expr("year(Date) * 100 + month(Date)")) \
    .withColumn("Month_Start_Date",date_format(expr("date_trunc('MM',Date)"), "yyyy-MM-dd")) \
    .withColumn("Month_End_Date",date_format(expr("last_day(date)"), "yyyy-MM-dd")) \
    .withColumn("Quarter_Name",expr("concat('Q', quarter(Date))")) \
    .withColumn("Quarter_Number",quarter(col("Date"))) \
    .withColumn("Quarter_Year_Number",expr("year(Date) * 10 + quarter(Date)")) \
    .withColumn("Quarter_Start_Date",date_format(expr("date_trunc('quarter', Date)"), "yyyy-MM-dd")) \
    .withColumn("Quarter_End_Date",date_format(expr("last_day(Date)"), "yyyy-MM-dd")) \
    .withColumn("Year",year(col("Date"))) \
    .withColumn("Year_Start_Date",date_format(expr("date_trunc('year', Date)"), "yyyy-MM-dd")) \
    .withColumn("Year_End_Date",date_format(expr("last_day(Date)"), "yyyy-MM-dd")) \
    .withColumn("Day_Seq",expr("row_number() over (order by Date)").cast(IntegerType())) \
    .withColumn("Week_Seq",expr("row_number() over (partion by weekofyear(Date) order by Date) - 1").cast(IntegerType())) \
    .withColumn("Year_Seq",expr("row_number() over (partion by year(Date) order by Date) - 1").cast(IntegerType()))

In [ ]:
# Add "Ago Date" Columns (for Example: 1 week ago, 1 month ago, etc)
calendar_df = calendar_df.withColumn("Week_Ago_Date", expr("date_sub(Date, 7)")) \
    .withColumn("Day_Ago_Date", expr("date_sub(Date, 1)")) \
    .withColumn("Month_Ago_Date", expr("add_months(Date, -1)")) \
    .withColumn("Quarte_Ago_Date", expr("add_months(Date, -3)")) \
    .withColumn("Year_Ago_Date", expr("add_months(Date, -12)")) 

In [ ]:
# Add "Day_Type" Column
calendar_df = calendar_df.withColumn("Day_Type",
                                     expr("CASE WHEN dayofweek(Date) IN (1, 7) THEN 'Weekend' ELSE 'Weekday' END"))

# Add placeholder for "Holiday_Name" (null as placeholder)
calendar_df = calendar_df.withColumn("Holiday_Name", lit(None).cast(IntegerType()))

# writing the final result
calendar_df.write.format("delta").mode("overwrite").saveAsTable("prepare.Calendar")